# OSFI Regulation Assistant: a RAG project
## Part 5: Run the packaged code as an API

Parts 1–4 were built and evaluated in notebooks. Part 5 runs the **packaged version** from your GitHub repository, the way it would run in production:

1. Get the code from GitHub
2. Run the **automated tests**
3. Build the **final search index** with all the evaluated fixes
4. Start the **REST API** (FastAPI) and call it like any other application would
5. Open the API's **interactive documentation** in your browser

**Before running:** upload the repository to GitHub (see the instructions in the chat), and set `REPO_URL` below.

**Runtime: T4 GPU.**

## Step 1: Get the code

In [ ]:
REPO_URL = "https://github.com/RubyaAfrin/osfi-rag-assistant"   # change if your repository has a different name

import os, sys
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/osfi_rag_project"
except Exception as e:
    print("Drive not available, using a local folder:", e)
    PROJECT_DIR = "osfi_rag_project"

PROJECT_DIR = os.path.abspath(PROJECT_DIR)
REPO_DIR = "/content/osfi-rag-assistant" if os.path.exists("/content") else os.path.abspath("osfi-rag-assistant")
if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
!git log --oneline -3

## Step 2: Install dependencies and run the tests

In [ ]:
!pip -q install -r requirements.txt -r requirements-test.txt
!python -m pytest -q

These are the same tests GitHub Actions runs every time you push a change. They use small stand-in models, so they finish in seconds, and several of them reproduce the exact bugs found in Part 4, so those bugs can't return unnoticed.

## Step 3: Build the final index

`scripts/build_index.py` takes the chunks from Part 2, applies the definition labels, and embeds everything. The final settings, stemming and a re-ranker pool of 50, are the defaults in the package.

In [ ]:
CHUNKS = os.path.join(PROJECT_DIR, "index", "chunks.jsonl")
FINAL_INDEX = os.path.join(PROJECT_DIR, "index_final")
!python scripts/build_index.py --chunks "{CHUNKS}" --out "{FINAL_INDEX}"

## Step 4: Start Ollama and the API

In [ ]:
!apt-get -qq install -y zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null

import subprocess, time, requests, json
ollama = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
!ollama pull qwen2.5:7b 2>&1 | tail -1

In [ ]:
subprocess.run(["pkill", "-f", "uvicorn osfi_rag.api"])   # stop a server left over from an earlier run
time.sleep(1)
env = dict(os.environ, OSFI_INDEX_DIR=FINAL_INDEX, OSFI_LLM_MODEL="qwen2.5:7b")
api_log = open("/tmp/api.log", "w")
api = subprocess.Popen(["uvicorn", "osfi_rag.api:app", "--host", "0.0.0.0", "--port", "8000"],
                       env=env, stdout=api_log, stderr=subprocess.STDOUT)
BASE = "http://localhost:8000"
for _ in range(60):
    try:
        print(requests.get(f"{BASE}/health", timeout=2).json()); break
    except Exception:
        time.sleep(2)
else:
    print("The API did not start. Its log:"); print(open("/tmp/api.log").read()[-3000:])

## Step 5: Call the API

This is exactly how another application, such as a website, a chatbot, or an internal tool, would use the assistant: by sending an HTTP request and receiving JSON. The first call loads the models, so it takes longer.

In [ ]:
def ask(question):
    r = requests.post(f"{BASE}/ask", json={"question": question}, timeout=300)
    r.raise_for_status()
    body = r.json()
    print(f"Q: {question}\n[{body['status'].upper()}] {body['answer']}")
    for s in body["sources"]:
        if s["cited"]:
            print(f"   [{s['n']}] {s['doc_id']} › {s['section']}")
    print(f"   ({body['seconds']}s)\n")
    return body

examples = [ask(q) for q in [
    "How does OSFI define a model in its model risk guideline?",
    "What is the maximum loan-to-value ratio for a HELOC?",
    "What is the minimum qualifying rate for uninsured mortgages?",
    "What does OSFI expect banks to do to manage cyber security risk?",
    "What is the best recipe for banana bread?",
]]

In [ ]:
print("The full JSON response for one question:")
print(json.dumps(examples[1], indent=2)[:2500])

In [ ]:
print("Search only, without the LLM:")
hits = requests.post(f"{BASE}/search", json={"query": "GDS and TDS ratio calculation", "k": 3}).json()
for h in hits:
    print(f"  {h['rank']}. {h['doc_id']} › {h['section'][:70]} (score {h['score']:.2f})")

## Step 6: Open the interactive API documentation

FastAPI automatically creates a web page that documents every endpoint and lets you try them. Run the next cell, then click the link it prints. On the page, open **POST /ask**, click **Try it out**, type a question, and click **Execute**.

In [ ]:
try:
    from google.colab.output import serve_kernel_port_as_window
    serve_kernel_port_as_window(8000, path="/docs")
except ImportError:
    print("Open http://localhost:8000/docs in your browser")

## Step 7: Save example responses for the README

In [ ]:
out = os.path.join(PROJECT_DIR, "evaluation", "api_examples.json")
json.dump(examples, open(out, "w"), indent=2)
print("Saved", out)

## Checkpoint

**Send me:**
1. **Step 2:** the test result line (for example, "26 passed")
2. **Step 3:** the "Saved ... chunks" line
3. **Step 5:** the five answers

Then we'll finish the README and add the project to your resumes.